# 05 — Scale the canonical decision landscape to 20 matches

**Scientific objective:** Transfer the verified single-match roster mapping and exact-frame joins across the full 20-match SkillCorner dataset. Use this notebook only to construct the *decision-level scaffold*, not to engineer pressing effectiveness, feasible carries, or pose features yet.

**Source keys:** `(match_id, period, tracking_frame, possession_event_id)`. Match-specific roster IDs are authoritative for player teams. Half-open `[start, end)` phase membership is an **analytical convention**. If an event-team ID disagrees with a phase possession-team ID, leave the phase annotation unassigned and flag it. These mismatches can occur *inside* a phase, not just at endpoints.

**Quality:** Retain positions and `is_detected` status separately. Carrier/team/frame consistency is a hard gate; phase-team disagreements and partial detections are explicit quality flags.

**Outputs:** `outputs/P1_all_matches_decision_gate.json` and 20 compact, local JSONL files. Upload **only the gate JSON**, not the full coordinate outputs.

**Patch 1:** Tracking frames can exist without a corresponding carrier record. These decisions are retained with `carrier_in_tracking_frame=False`, `carrier_xy=None`, and an explicit per-match count. They must be excluded from carrier-location-dependent analysis.


In [1]:
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime, timezone
import csv, json, math

ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "data" / "matches").is_dir()), None)
if ROOT is None: raise FileNotFoundError("Run notebook inside the SkillCorner repository.")
MATCHES = ROOT / "data" / "matches"
OUT = ROOT / "outputs"
OUT.mkdir(exist_ok=True)
MATCH_IDS = sorted(p.name for p in MATCHES.iterdir() if p.is_dir() and p.name.isdigit())
assert len(MATCH_IDS) == 20, f"Expected 20 local matches, found {len(MATCH_IDS)}"
print("Working with", len(MATCH_IDS), "matches; outputs:", OUT)


Working with 20 matches; outputs: /Users/fadihaddad/Documents/GitHub/SkillCorner_opendata_soccer/outputs


## 1. Reusable, explicit parsing rules

Event labels such as `carry` and `end_type` reflect realized actions and outcomes. They are preserved under a separate `outcomes` object and **must not** be passed to a pre-action model as features. Overlapping event IDs are diagnostic references only, not validated pre-action attributes.


**Roster exceptions:** A carrier that is absent from `match.json` is not equivalent to an incorrect team assignment. We retain that decision and label its carrier roster status as `missing`, while requiring its event team to be one of the two official match teams. A *conflicting* team assignment for a rostered player still stops execution. Absent roster identities cannot be used to infer unknown players' teams.

In [2]:
def read_csv(p):
    with p.open(newline="", encoding="utf-8-sig") as f:
        return list(csv.DictReader(f))
def integer(v):
    try: return int(v)
    except (ValueError, TypeError): return None
def textval(v):
    return None if v is None or str(v).strip() == "" else str(v).strip()
def finite_xy(p):
    if not isinstance(p, dict): return None
    try:
        x, y = float(p["x"]), float(p["y"])
        return [x,y] if math.isfinite(x) and math.isfinite(y) else None
    except (KeyError, TypeError, ValueError): return None

def match_one(mid):
    folder = MATCHES / mid
    meta = json.loads((folder / f"{mid}_match.json").read_text())
    raw_events = read_csv(folder / f"{mid}_dynamic_events.csv")
    phase_data = read_csv(folder / f"{mid}_phases_of_play.csv")
    roster = {str(p["id"]):str(p["team_id"]) for p in meta["players"]
              if p.get("id") is not None and p.get("team_id") is not None}
    assert len(set(roster.values())) == 2, f"{mid}: roster team count != 2"
    possessions = [e for e in raw_events if e.get("event_type") == "player_possession"]
    phases_by_period = defaultdict(list)
    for p in phase_data:
        a,b,period = integer(p.get("frame_start")),integer(p.get("frame_end")),integer(p.get("period"))
        if None not in (a,b,period):
            phases_by_period[period].append((a,b,p))
    # Distinguish roster absence from an actual contradiction.
    # All event team IDs must be one of the match's two documented teams.
    valid_teams = {str(meta["home_team"]["id"]),str(meta["away_team"]["id"])}
    for e in possessions:
        pid,team = textval(e.get("player_id")),textval(e.get("team_id"))
        if team not in valid_teams:
            raise AssertionError(f"{mid}: carrier {pid} has unknown event team {team}")
        if pid in roster and roster[pid] != team:
            raise AssertionError(f"{mid}: carrier {pid} has conflicting event/roster teams {team} vs {roster[pid]}")
    rows = []
    for e in possessions:
        period,frame = integer(e.get("period")),integer(e.get("frame_start"))
        if period is None or frame is None:
            raise AssertionError(f"{mid}: possession missing period/frame")
        candidates = [p for a,b,p in phases_by_period[period] if a<=frame<b]
        status = ("missing" if not candidates else "multiple") if len(candidates)!=1 else (
            "ok" if textval(candidates[0].get("team_in_possession_id"))==textval(e.get("team_id"))
            else "team_disagreement")
        phase = candidates[0] if status == "ok" else None
        rows.append({
            "match_id":mid,"period":period,"frame":frame,
            "possession_event_id":textval(e.get("event_id")),
            "carrier_player_id":textval(e.get("player_id")),
            "attacking_team_id":textval(e.get("team_id")),
            "carrier_roster_status":("matched" if textval(e.get("player_id")) in roster else "missing"),
            "phase_status":status,
            "phase_index":integer(phase.get("index")) if phase else None,
            "defending_phase_annotation":textval(phase.get("team_out_of_possession_phase_type")) if phase else None,
            "outcomes":{"carry_realized":textval(e.get("carry")),"end_type":textval(e.get("end_type"))},
            "teammates_detected":[],"teammates_extrapolated":[],
            "opponents_detected":[],"opponents_extrapolated":[],
            "carrier_xy":None,"carrier_detected":None,"carrier_in_tracking_frame":False,"ball_xy":None,"ball_detected":None,
            "unresolved_players":0
        })
    wanted={(r["period"],r["frame"]) for r in rows}
    if len(wanted)!=len(rows):
        raise AssertionError(f"{mid}: multiple possessions at same period/frame; inspect before deduplication")
    row_by_key={(r["period"],r["frame"]):r for r in rows}
    matched=set()
    skipped_null_keys=0
    with (folder / f"{mid}_tracking_extrapolated.jsonl").open(encoding="utf-8") as f:
        for line in f:
            t=json.loads(line)
            period,frame=integer(t.get("period")),integer(t.get("frame"))
            if period is None or frame is None:
                skipped_null_keys+=1
                continue
            key=(period,frame)
            if key not in row_by_key:
                continue
            if key in matched:
                raise AssertionError(f"{mid}: duplicate requested tracking key {key}")
            matched.add(key)
            r=row_by_key[key]
            carrier_id=r["carrier_player_id"]
            team_id=r["attacking_team_id"]
            pdata=t.get("player_data") or []
            player_index={textval(p.get("player_id")):p for p in pdata}
            carrier=player_index.get(carrier_id)
            # A valid frame may omit the player named by the possession event.
            # Preserve the decision and signal spatial ineligibility explicitly.
            r["carrier_in_tracking_frame"] = carrier is not None
            if carrier is not None:
                r["carrier_xy"] = finite_xy(carrier)
                r["carrier_detected"] = carrier.get("is_detected") is True
            ball=t.get("ball_data") or {}
            r["ball_xy"]=finite_xy(ball)
            r["ball_detected"]=ball.get("is_detected") is True
            for p in pdata:
                pid=textval(p.get("player_id"))
                if pid==carrier_id:continue
                pt=finite_xy(p)
                if pt is None:continue
                other_team=roster.get(pid)
                if other_team is None:
                    r["unresolved_players"]+=1
                    continue
                role="teammates" if other_team==team_id else "opponents"
                quality="detected" if p.get("is_detected") is True else "extrapolated"
                r[f"{role}_{quality}"].append({"player_id":pid,"xy":pt})
            if len(matched)==len(wanted):break
    if len(matched)!=len(wanted):
        missing=sorted(wanted-matched)
        raise AssertionError(f"{mid}: {len(missing)} tracking keys missing; examples {missing[:5]}")
    # Gate counts after processing all requested frames.
    gate={
        "match_id":mid,"decisions":len(rows),"roster_players":len(roster),
        "tracking_frames_found":len(matched),"skipped_tracking_records_with_invalid_key":skipped_null_keys,
        "carrier_roster_missing":sum(r["carrier_roster_status"]=="missing" for r in rows),
        "carrier_roster_missing_examples":[
            {"period":r["period"],"frame":r["frame"],"carrier_player_id":r["carrier_player_id"],
             "event_team_id":r["attacking_team_id"]}
            for r in rows if r["carrier_roster_status"]=="missing"][:10],
        "phase_status":dict(Counter(r["phase_status"] for r in rows)),
        "carrier_present_in_tracking":sum(r["carrier_in_tracking_frame"] for r in rows),
        "carrier_missing_from_tracking":sum(not r["carrier_in_tracking_frame"] for r in rows),
        "missing_carrier_examples":[
            {"period":r["period"],"frame":r["frame"],"possession_event_id":r["possession_event_id"],"carrier_player_id":r["carrier_player_id"]}
            for r in rows if not r["carrier_in_tracking_frame"]][:10],
        "carrier_xy_valid":sum(r["carrier_xy"] is not None for r in rows),
        "carrier_detected":sum(r["carrier_detected"] is True for r in rows),
        "ball_xy_valid":sum(r["ball_xy"] is not None for r in rows),
        "rows_with_unresolved_players":sum(r["unresolved_players"]>0 for r in rows)
    }
    output=OUT/f"P1_canonical_decisions_match_{mid}_ROLES.jsonl"
    with output.open("w",encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r,separators=(",",":"))+"\n")
    return gate


## 2. Execute on all 20 matches

This reads each match's tracking JSONL **once**, retaining only the exact frames needed for player-possession starts. It does not load pose archives or compute expensive spatial models. A failed hard gate stops the run rather than producing a superficially complete dataset.

**Missing carrier records are now a flagged limitation, not a fatal error.** The notebook still halts for missing or duplicate requested tracking keys and roster/team inconsistencies.


In [3]:
results=[]
for mid in MATCH_IDS:
    info=match_one(mid)
    results.append(info)
    print(f"{mid}: {info['decisions']} decisions, "
          f"{info['tracking_frames_found']} frames, "
          f"{info['phase_status']}, "
          f"{info['rows_with_unresolved_players']} rows with unresolved players")


1874553: 959 decisions, 959 frames, {'ok': 950, 'team_disagreement': 9}, 0 rows with unresolved players
1886347: 999 decisions, 999 frames, {'ok': 979, 'team_disagreement': 18, 'missing': 2}, 0 rows with unresolved players
1899585: 929 decisions, 929 frames, {'ok': 909, 'team_disagreement': 20}, 0 rows with unresolved players
1925299: 1062 decisions, 1062 frames, {'ok': 1051, 'team_disagreement': 10, 'missing': 1}, 0 rows with unresolved players
1927964: 1140 decisions, 1140 frames, {'ok': 1128, 'team_disagreement': 9, 'missing': 3}, 0 rows with unresolved players
1953632: 979 decisions, 979 frames, {'ok': 967, 'team_disagreement': 9, 'missing': 3}, 0 rows with unresolved players
1959846: 946 decisions, 946 frames, {'ok': 939, 'team_disagreement': 6, 'missing': 1}, 0 rows with unresolved players
1986691: 915 decisions, 915 frames, {'ok': 893, 'team_disagreement': 21, 'missing': 1}, 0 rows with unresolved players
1996435: 1005 decisions, 1005 frames, {'ok': 980, 'team_disagreement': 23,

## 3. Summarize reproducible data eligibility

Rows with a phase-team disagreement retain their player identities and tracking coordinates, but **their defensive phase is unassigned**. A team change within an interval might reflect a brief turnover or the coarser semantics of a phase classification. We do not alter SkillCorner's annotations or choose a label without evidence. Summary counts below establish sample availability, **not model performance**.


In [4]:
summary={
    "created_utc":datetime.now(timezone.utc).isoformat(),
    "n_matches":len(results),
    "decisions":sum(x["decisions"] for x in results),
    "tracking_frames_found":sum(x["tracking_frames_found"] for x in results),
    "carrier_present_in_tracking":sum(x["carrier_present_in_tracking"] for x in results),
    "carrier_missing_from_tracking":sum(x["carrier_missing_from_tracking"] for x in results),
    "carrier_xy_valid":sum(x["carrier_xy_valid"] for x in results),
    "carrier_roster_missing":sum(x["carrier_roster_missing"] for x in results),
    "carrier_missing_from_tracking":sum(x["carrier_missing_from_tracking"] for x in results),
    "carrier_detected":sum(x["carrier_detected"] for x in results),
    "ball_xy_valid":sum(x["ball_xy_valid"] for x in results),
    "phase_status":dict(sum((Counter(x["phase_status"]) for x in results),Counter())),
    "rows_with_unresolved_players":sum(x["rows_with_unresolved_players"] for x in results),
    "notes":[
      "20-match canonical decision-level scaffold; no predictive features trained",
      "Tracking player roles from match roster players[].id and team_id",
      "Half-open phase membership is an analytical convention, not source-verified",
      "Phase mismatches left unassigned; do not use as phase-labeled observations",
      "Missing carrier tracking records retained with carrier_in_tracking_frame=false and carrier_xy=null; not spatially eligible",
      "Detected and extrapolated XY preserved; finite XY does not guarantee reliable detection",
      "Outcomes separated from time-zero spatial observations",
      "EPV fields intentionally not required; full-body pose intentionally deferred",
      "Full per-match coordinate outputs remain local"]
}
report={"summary":summary,"matches":results}
dest=OUT/"P1_all_matches_decision_gate.json"
dest.write_text(json.dumps(report,indent=2),encoding="utf-8")
print(json.dumps(summary,indent=2))
print("UPLOAD ONLY:",dest)


{
  "created_utc": "2026-10-24T23:44:55.674314+00:00",
  "n_matches": 20,
  "decisions": 18792,
  "tracking_frames_found": 18792,
  "carrier_present_in_tracking": 18757,
  "carrier_missing_from_tracking": 35,
  "carrier_xy_valid": 18757,
  "carrier_roster_missing": 1,
  "carrier_detected": 18092,
  "ball_xy_valid": 18758,
  "phase_status": {
    "ok": 18474,
    "team_disagreement": 288,
    "missing": 30
  },
  "rows_with_unresolved_players": 0,
  "notes": [
    "20-match canonical decision-level scaffold; no predictive features trained",
    "Tracking player roles from match roster players[].id and team_id",
    "Half-open phase membership is an analytical convention, not source-verified",
    "Phase mismatches left unassigned; do not use as phase-labeled observations",
    "Missing carrier tracking records retained with carrier_in_tracking_frame=false and carrier_xy=null; not spatially eligible",
    "Detected and extrapolated XY preserved; finite XY does not guarantee reliable dete